# Wildlife Strike Prediction — Complete Pipeline (GPU)

Phases 2–5: Data Cleaning → Feature Engineering → Modeling → Submission

**Runtime:** Set to **GPU** (Runtime → Change runtime type → T4 GPU) for XGBoost and PyTorch acceleration.

In [ ]:
# --- Colab Setup: install dependencies & mount Google Drive ---
!pip install -q imbalanced-learn xgboost lightgbm pyarrow

from google.colab import drive, files as colab_files
import os

drive.mount('/content/drive')

# ── Set your Drive folder path here ──
DRIVE_FOLDER = '/content/drive/MyDrive/Kaggle'   # <-- edit if your CSVs are in a subfolder

train_path = os.path.join(DRIVE_FOLDER, 'train.csv')
test_path  = os.path.join(DRIVE_FOLDER, 'test.csv')

assert os.path.exists(train_path), f'train.csv not found at {train_path}'
assert os.path.exists(test_path),  f'test.csv not found at {test_path}'
print(f'train.csv: {os.path.getsize(train_path)/1e6:.1f} MB')
print(f'test.csv:  {os.path.getsize(test_path)/1e6:.1f} MB')

ModuleNotFoundError: No module named 'google'

## Phases 2–3: Data Cleaning & Feature Engineering

We define a reusable `feature_engineer()` function that applies all non-target-dependent
transforms to both training and test data identically. This guarantees column alignment
and prevents train/test skew.

**Feature groups:**
1. Missingness flags — whether a field is blank is itself a damage signal
2. Temporal / cyclical encodings — month, season, migration period
3. Flight phase risk — ordinal risk score from FAA domain knowledge
4. Wildlife quantity & size — flock encoding, struck-to-seen ratio
5. Aircraft type / weather / geospatial — jet flag, coast proximity, adverse weather
6. Physical interaction terms & polynomials — kinetic energy proxies
7. Target encoding — species/airport/state damage rates (OOF, leak-free) *[separate step]*
8. Impact severity proxies — altitude risk zones, bird kinetic energy
9. Multi-engine vulnerability & triple interactions

In [ ]:
import pandas as pd, numpy as np, warnings, gc, re
warnings.filterwarnings("ignore")

def feature_engineer(df, null_drop_cols):
    """Apply Phase 2-3 feature engineering (all non-target-dependent transforms).
    
    Parameters
    ----------
    df : DataFrame - raw train or test data
    null_drop_cols : list - columns with >50% nulls in TRAINING data
    """
    df = df.copy()

    # ---- Phase 2: Cleaning ----

    # Mine REMARKS / COMMENTS for damage keywords BEFORE dropping them.
    # ~50% of rows have text; non-null entries strongly correlate with damage.
    for text_col in ['REMARKS', 'COMMENTS']:
        if text_col not in df.columns:
            continue
        txt = df[text_col].fillna('').astype(str).str.lower()
        p = text_col[:3].lower()  # 'rem' or 'com'
        df[f'has_{p}']       = (txt.str.len() > 0).astype(np.int8)
        df[f'{p}_len']       = txt.str.len().clip(upper=500)
        df[f'{p}_engine']    = txt.str.contains('engine|ingest|turbine|fan blade', regex=True).astype(np.int8)
        df[f'{p}_struct']    = txt.str.contains('dent|crack|hole|shatter|damage|broke', regex=True).astype(np.int8)
        df[f'{p}_windshld']  = txt.str.contains('windshield|canopy|window|radome', regex=True).astype(np.int8)
        df[f'{p}_evidence']  = txt.str.contains('blood|feather|remains|carcass', regex=True).astype(np.int8)
        df[f'{p}_kw_count']  = (df[f'{p}_engine'] + df[f'{p}_struct'] +
                                df[f'{p}_windshld'] + df[f'{p}_evidence'])

    drops = list(set(null_drop_cols + [
        'INDEX_NR','REMARKS','COMMENTS','LUPDATE','TRANSFER',
        'BIRD_BAND_NUMBER','AIRPORT','INCIDENT_DATE'
    ]))
    df.drop(columns=[c for c in drops if c in df.columns], inplace=True)

    # Missingness flags (before imputation)
    for col in ['SPEED','HEIGHT','DISTANCE','TIME_DECIMAL','LATITUDE']:
        if col in df.columns:
            df[f'MISS_{col}'] = df[col].isna().astype('int8')
    mf = [c for c in df.columns if c.startswith('MISS_')]
    if mf:
        df['MISS_total'] = df[mf].sum(axis=1)

    for col in ['LATITUDE','LONGITUDE']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Group-based median imputation
    if 'AC_MASS' in df.columns and 'SPEED' in df.columns:
        df['SPEED'] = df.groupby('AC_MASS')['SPEED'].transform(lambda x: x.fillna(x.median()))
    if 'AC_MASS' in df.columns and 'HEIGHT' in df.columns:
        df['HEIGHT'] = df.groupby('AC_MASS')['HEIGHT'].transform(lambda x: x.fillna(x.median()))

    num_cols = df.select_dtypes(include=['float64','int64']).columns.tolist()
    num_cols = [c for c in num_cols if c != 'INDICATED_DAMAGE']
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    for col in ['SPEED','HEIGHT','DISTANCE']:
        if col in df.columns:
            lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
            df[col] = df[col].clip(lower=lo, upper=hi)

    # ---- Phase 3: Feature Engineering ----

    # Group 2: Temporal
    MIG = {8, 9, 10}

    # Parse TIME (HH:MM string) into decimal hours — was missing before!
    if 'TIME' in df.columns and 'TIME_DECIMAL' not in df.columns:
        def _parse_hhmm(v):
            if pd.isna(v): return np.nan
            m = re.match(r'(\d{1,2}):(\d{2})', str(v).strip())
            return float(m.group(1)) + float(m.group(2)) / 60.0 if m else np.nan
        df['hour_decimal'] = df['TIME'].apply(_parse_hhmm)
        h = df['hour_decimal'].fillna(12.0)
        df['hour_sin'] = np.sin(2 * np.pi * h / 24)
        df['hour_cos'] = np.cos(2 * np.pi * h / 24)
        df['is_night_strike'] = ((h >= 20) | (h < 6)).astype(np.int8)
        df['is_rush_strike']  = ((h >= 6) & (h <= 9)).astype(np.int8)
        df['is_dawn_dusk']    = (((h >= 5) & (h < 7)) | ((h >= 18) & (h < 20))).astype(np.int8)
        df.drop(columns=['hour_decimal'], inplace=True)

    if 'TIME_DECIMAL' in df.columns:
        h = df['TIME_DECIMAL'].fillna(12.0)
        df['hour_sin'] = np.sin(2 * np.pi * h / 24)
        df['hour_cos'] = np.cos(2 * np.pi * h / 24)
        df['is_night_strike'] = ((h >= 20) | (h < 6)).astype(np.int8)
        df['is_rush_strike']  = ((h >= 6) & (h <= 9)).astype(np.int8)
        df['is_dawn_dusk']    = (((h >= 5) & (h < 7)) | ((h >= 18) & (h < 20))).astype(np.int8)
        df.drop(columns=['TIME_DECIMAL'], inplace=True)
    if 'INCIDENT_MONTH' in df.columns:
        m = df['INCIDENT_MONTH'].fillna(6)
        df['month_sin'] = np.sin(2 * np.pi * m / 12)
        df['month_cos'] = np.cos(2 * np.pi * m / 12)
        df['is_migration_season'] = m.isin(MIG).astype(np.int8)
        df['season'] = pd.cut(m, bins=[0,3,6,9,12], labels=[1,2,3,4],
                              include_lowest=True).astype(float)
    if 'INCIDENT_YEAR' in df.columns:
        df['year_trend'] = (df['INCIDENT_YEAR'] - 1990).clip(lower=0)

    # Group 3: Phase risk
    PR = {'Parked':0.5,'Taxi':1.0,'Local':1.5,'Arrival':2.0,'En Route':2.0,
          'Descent':2.5,'Departure':2.5,'Climb':3.0,'Take-off Run':4.0,
          'Landing Roll':4.5,'Approach':5.0}
    if 'PHASE_OF_FLIGHT' in df.columns:
        df['phase_risk'] = df['PHASE_OF_FLIGHT'].map(PR).fillna(2.5)
        df['is_ground_phase'] = df['PHASE_OF_FLIGHT'].isin(
            {'Taxi','Parked','Landing Roll','Take-off Run'}).astype(np.int8)
        df['is_high_risk_phase'] = df['PHASE_OF_FLIGHT'].isin(
            {'Approach','Take-off Run','Landing Roll','Climb'}).astype(np.int8)
        df.drop(columns=['PHASE_OF_FLIGHT'], inplace=True)

    # Group 4: Wildlife
    RM = {'1':1.0,'10-Feb':5.0,'11-100':30.0,'More than 100':150.0}
    for col, new in [('NUM_SEEN','num_seen'),('NUM_STRUCK','num_struck')]:
        if col in df.columns:
            df[new] = df[col].astype(str).str.strip().map(RM).fillna(1.0)
            df.drop(columns=[col], inplace=True)
        else:
            df[new] = 1.0
    if 'SIZE' in df.columns:
        df['size_ordinal'] = df['SIZE'].map({'Small':1,'Medium':2,'Large':3}).fillna(1.0)
        df.drop(columns=['SIZE'], inplace=True)
    else:
        df['size_ordinal'] = 1.0
    df['is_flock'] = (df['num_seen'] > 1).astype(np.int8)
    df['is_large_flock'] = (df['num_seen'] >= 30).astype(np.int8)
    df['struck_seen_ratio'] = (df['num_struck'] /
                                df['num_seen'].replace(0, np.nan)).fillna(1.0)

    # Group 5: Aircraft / Weather / Geo
    if 'TYPE_ENG' in df.columns:
        df['is_jet'] = df['TYPE_ENG'].str.strip().isin({'B','D'}).astype(np.int8)
        df.drop(columns=['TYPE_ENG'], inplace=True)
    if 'AC_CLASS' in df.columns:
        df['is_helicopter'] = (df['AC_CLASS'].str.strip() == 'B').astype(np.int8)
        df.drop(columns=['AC_CLASS'], inplace=True)
    if 'AC_MASS' in df.columns:
        df['AC_MASS'] = pd.to_numeric(df['AC_MASS'], errors='coerce').fillna(2.0)
        df['is_heavy_aircraft'] = (df['AC_MASS'] >= 4).astype(np.int8)
    if 'WARNED' in df.columns:
        df['warned_yes'] = (df['WARNED'] == 'Yes').astype(np.int8)
        df['warned_no']  = (df['WARNED'] == 'No').astype(np.int8)
        df.drop(columns=['WARNED'], inplace=True)
    if 'TIME_OF_DAY' in df.columns:
        df['light_level'] = df['TIME_OF_DAY'].map(
            {'Night':0,'Dawn':1,'Dusk':1,'Day':3}).fillna(1.5)
        df.drop(columns=['TIME_OF_DAY'], inplace=True)
    if 'SKY' in df.columns:
        df['sky_score'] = df['SKY'].map(
            {'No Cloud':0,'Some Cloud':1,'Overcast':2}).fillna(0.0)
        df.drop(columns=['SKY'], inplace=True)
    PW = {'Rain','Fog','Snow','Hail'}
    if 'PRECIPITATION' in df.columns:
        df['has_precip'] = df['PRECIPITATION'].apply(
            lambda v: 0 if pd.isna(v) else int(any(k in str(v) for k in PW))).astype(np.int8)
        df['precip_count'] = df['PRECIPITATION'].apply(
            lambda v: 0 if pd.isna(v) else sum(k in str(v) for k in PW)).astype(np.int8)
        df.drop(columns=['PRECIPITATION'], inplace=True)
    for col in ['LATITUDE','LONGITUDE']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'LATITUDE' in df.columns and 'LONGITUDE' in df.columns:
        lat = df['LATITUDE'].fillna(37.0)
        lon = df['LONGITUDE'].fillna(-96.0)
        df['dist_east_coast'] = (lon + 75.0).abs()
        df['dist_west_coast'] = (lon + 120.0).abs()
        df['dist_gulf_coast'] = (lat - 29.0).abs()
        df['dist_us_center']  = np.sqrt((lat - 39.5)**2 + (lon + 98.35)**2)
        df['is_northern'] = (lat >= 42.0).astype(np.int8)
        df['is_coastal']  = ((df['dist_east_coast'] < 3) |
                             (df['dist_west_coast'] < 3)).astype(np.int8)

    # Group 5b: Species taxonomy (keyword-based biological grouping)
    if 'SPECIES' in df.columns:
        sp_str = df['SPECIES'].fillna('').astype(str).str.lower()
        df['sp_waterfowl'] = sp_str.str.contains(
            'goose|geese|duck|swan|pelican|crane|heron|egret|cormorant', regex=True).astype(np.int8)
        df['sp_raptor'] = sp_str.str.contains(
            'hawk|eagle|falcon|owl|vulture|osprey|kite|kestrel|harrier', regex=True).astype(np.int8)
        df['sp_gull'] = sp_str.str.contains('gull|tern|skimmer', regex=True).astype(np.int8)
        df['sp_unknown'] = sp_str.str.contains('unknown', regex=True).astype(np.int8)
        df['sp_large_danger'] = ((df['sp_waterfowl'] | df['sp_raptor']) &
                                  (df['size_ordinal'] >= 2)).astype(np.int8)
        df['sp_waterfowl_jet'] = (df['sp_waterfowl'] * df['is_jet']).astype(np.int8)
        df['sp_raptor_jet']    = (df['sp_raptor'] * df['is_jet']).astype(np.int8)

    # Group 6: Interactions & polynomials
    sp = pd.to_numeric(df.get('SPEED',  pd.Series(0, index=df.index)), errors='coerce').fillna(0)
    ht = pd.to_numeric(df.get('HEIGHT', pd.Series(0, index=df.index)), errors='coerce').fillna(0)
    ms = pd.to_numeric(df.get('AC_MASS',pd.Series(2, index=df.index)), errors='coerce').fillna(2)
    df['kinetic_proxy']     = ms * sp ** 2
    df['speed_x_height']    = sp * ht
    df['strike_mass_index'] = df['size_ordinal'] * df['num_struck']
    df['risk_composite']    = df['phase_risk'] * df['size_ordinal'] * np.log1p(df['num_struck'])
    df['jet_x_size']  = df['is_jet'] * df['size_ordinal']
    df['jet_x_flock'] = df['is_jet'] * df['is_flock']
    if 'has_precip' in df.columns and 'light_level' in df.columns:
        df['poor_visibility'] = df['has_precip'] * (3 - df['light_level']).clip(lower=0)
    df['ground_x_flock'] = df['is_ground_phase'] * df['is_flock']
    for col in ['SPEED','HEIGHT','DISTANCE']:
        if col in df.columns:
            raw = pd.to_numeric(df[col], errors='coerce').clip(lower=0)
            df[f'{col}_log1p'] = np.log1p(raw)
            df[f'{col}_sqrt']  = np.sqrt(raw.fillna(0))
            df[f'{col}_sq']    = raw.fillna(0) ** 2

    # Group 8: Impact severity
    for col in ['REMAINS_COLLECTED','REMAINS_SENT']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(np.int8)
    sp = pd.to_numeric(df.get('SPEED',  pd.Series(0, index=df.index)), errors='coerce').fillna(0)
    ht = pd.to_numeric(df.get('HEIGHT', pd.Series(0, index=df.index)), errors='coerce').fillna(0)
    df['bird_kinetic']    = df['size_ordinal'] ** 2 * sp
    df['flock_impact']    = df['num_struck'] * df['size_ordinal'] ** 2 * sp
    df['alt_ground']      = (ht == 0).astype(np.int8)
    df['alt_very_low']    = ((ht > 0) & (ht <= 200)).astype(np.int8)
    df['alt_low']         = ((ht > 200) & (ht <= 500)).astype(np.int8)
    df['alt_medium']      = ((ht > 500) & (ht <= 2000)).astype(np.int8)
    df['height_inv']      = 1.0 / (ht + 1.0)
    df['speed_very_high'] = (sp >= 250).astype(np.int8)
    df['speed_high']      = ((sp >= 150) & (sp < 250)).astype(np.int8)
    df['speed_low']       = (sp < 100).astype(np.int8)

    # Group 9: Multi-engine & triple interactions
    sp = pd.to_numeric(df.get('SPEED', pd.Series(0, index=df.index)), errors='coerce').fillna(0)
    if 'NUM_ENGS' in df.columns:
        df['NUM_ENGS'] = pd.to_numeric(df['NUM_ENGS'], errors='coerce').fillna(1)
        df['is_multi_engine'] = (df['NUM_ENGS'] >= 2).astype(np.int8)
        df['multi_eng_jet']   = df['is_multi_engine'] * df['is_jet']
        df['engs_x_size']     = df['NUM_ENGS'] * df['size_ordinal']
    df['deadly_combo'] = (df['is_jet'] * (df['size_ordinal'] >= 2).astype(np.int8) *
                          df['is_high_risk_phase'])
    df['size_x_speed']    = df['size_ordinal'] * sp
    df['size_x_phase']    = df['size_ordinal'] * df['phase_risk']
    df['size_x_alt_inv']  = df['size_ordinal'] * df['height_inv']
    df['jet_approach_large'] = (df['is_jet'] *
                                (df['size_ordinal'] == 3).astype(np.int8) *
                                (df['phase_risk'] >= 4.0).astype(np.int8))
    df['fast_and_low'] = (df['speed_very_high'] *
                          (df['alt_ground'] | df['alt_very_low']).astype(np.int8))

    return df

print('feature_engineer() defined.')

In [ ]:
# --- Load raw data & apply feature engineering ---
train_df = pd.read_csv(train_path, low_memory=False)
test_df  = pd.read_csv(test_path,  low_memory=False)
submission_ids = test_df['INDEX_NR'].copy()
print(f'Raw — Train: {train_df.shape}, Test: {test_df.shape}')

null_drop_cols = train_df.columns[train_df.isnull().mean() > 0.50].tolist()

df_train = feature_engineer(train_df, null_drop_cols)
df_test  = feature_engineer(test_df,  null_drop_cols)
del train_df, test_df; gc.collect()
print(f'After FE — Train: {df_train.shape}, Test: {df_test.shape}')

# --- Group 7: Smoothed Target Encoding (OOF for train, full-map for test) ---
from sklearn.model_selection import StratifiedKFold

SMOOTH = 30
y_full = df_train['INDICATED_DAMAGE']
gm = float(y_full.mean())
te_maps_p3 = {}

# Create a species group column for target encoding (lower cardinality than raw SPECIES)
for df in [df_train, df_test]:
    if 'SPECIES' in df.columns:
        sp_str = df['SPECIES'].fillna('Unknown').astype(str).str.lower()
        conditions = [
            sp_str.str.contains('goose|geese|duck|swan|pelican|crane|heron|egret|cormorant'),
            sp_str.str.contains('hawk|eagle|falcon|owl|vulture|osprey|kite|kestrel'),
            sp_str.str.contains('gull|tern|skimmer'),
            sp_str.str.contains('sparrow|finch|swallow|starling|blackbird|robin|wren|warbler'),
            sp_str.str.contains('pigeon|dove'),
            sp_str.str.contains('bat'),
        ]
        choices = ['waterfowl', 'raptor', 'gull', 'songbird', 'pigeon', 'bat']
        df['SPECIES_GROUP'] = np.select(conditions, choices, default='other')

TE_COLS = ['SPECIES','SPECIES_GROUP','AIRPORT_ID','STATE','OPID','FAAREGION']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in TE_COLS:
    if col not in df_train.columns:
        continue
    oof = pd.Series(np.nan, index=df_train.index)
    for ti, vi in skf.split(df_train, y_full):
        cats = df_train[col].iloc[ti].astype(str).fillna('Unknown')
        ys = y_full.iloc[ti]
        grp = pd.DataFrame({'cat': cats, 'y': ys}).groupby('cat')['y'].agg(['sum','count'])
        grp['enc'] = (grp['sum'] + SMOOTH * gm) / (grp['count'] + SMOOTH)
        oof.iloc[vi] = (df_train[col].iloc[vi].astype(str).fillna('Unknown')
                        .map(grp['enc'].to_dict()).fillna(gm).values)
    df_train[f'{col}_te'] = oof
    cats_all = df_train[col].astype(str).fillna('Unknown')
    grp_all = pd.DataFrame({'cat': cats_all, 'y': y_full}).groupby('cat')['y'].agg(['sum','count'])
    grp_all['enc'] = (grp_all['sum'] + SMOOTH * gm) / (grp_all['count'] + SMOOTH)
    te_maps_p3[col] = grp_all['enc'].to_dict()
    df_test[f'{col}_te'] = (df_test[col].astype(str).fillna('Unknown')
                            .map(te_maps_p3[col]).fillna(gm))
    df_train.drop(columns=[col], inplace=True)
    df_test.drop(columns=[col], inplace=True)
    print(f'  {col:12s} → {col}_te  (range {oof.min():.3f} – {oof.max():.3f})')

# NUM_ENGS target encoding
if 'NUM_ENGS' in df_train.columns:
    cats = df_train['NUM_ENGS'].astype(str)
    grp = pd.DataFrame({'cat': cats, 'y': y_full}).groupby('cat')['y'].agg(['sum','count'])
    grp['enc'] = (grp['sum'] + SMOOTH * gm) / (grp['count'] + SMOOTH)
    te_maps_p3['NUM_ENGS'] = grp['enc'].to_dict()
    df_train['NUM_ENGS_te'] = cats.map(te_maps_p3['NUM_ENGS']).fillna(gm)
    df_test['NUM_ENGS_te'] = df_test['NUM_ENGS'].astype(str).map(te_maps_p3['NUM_ENGS']).fillna(gm)

# Group 10: Cross TE & risk concentration
for df in [df_train, df_test]:
    if 'SPECIES_te' in df.columns:
        df['species_x_state_risk']   = df['SPECIES_te'] * df['STATE_te'] / gm
        df['species_x_airport_risk'] = df['SPECIES_te'] * df['AIRPORT_ID_te'] / gm
        df['species_x_phase']        = df['SPECIES_te'] * df['phase_risk']
        df['airport_x_season']       = df['AIRPORT_ID_te'] * df['season']
    rf = ['is_jet','is_high_risk_phase','is_flock','speed_very_high','alt_very_low','alt_ground']
    df['risk_flag_count'] = df[[f for f in rf if f in df.columns]].sum(axis=1).astype(np.int8)

print(f'\nAfter TE — Train: {df_train.shape}, Test: {df_test.shape}')

## Phase 4: Modeling & Hyperparameter Tuning

We split the training data 80/20 for initial model development, encode remaining categorical
columns (OOF target encoding + frequency + ordinal), and scale for linear/MLP models.

**Individual model cells** (below) train each model once for quick iteration and narrative:
1. **Logistic Regression** — linear baseline
2. **XGBoost** (GPU) — second-order gradient boosting
3. **PyTorch MLP** (GPU) — 5-layer feed-forward network with dropout & GELU
4. **HistGradientBoosting** — sklearn histogram-based ensemble
5. **LightGBM** (GPU) — leaf-wise boosting
6. **XGBoost (tuned)** — RandomizedSearchCV over hyperparameter space

**Final ensemble** uses K-fold OOF stacking with a LogisticRegression meta-learner
(see stacking cell below), then retrains all models on 100% data for submission.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, balanced_accuracy_score

y_all = df_train['INDICATED_DAMAGE']
X_all = df_train.drop(columns=['INDICATED_DAMAGE'])
X_sub = df_test.copy()  # competition test set
del df_train, df_test; gc.collect()

X_trn, X_val, y_trn, y_val = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)

# Identify remaining categorical columns
obj_cols = X_trn.select_dtypes(include=['object','string','category']).columns.tolist()
print(f'Categorical columns to encode ({len(obj_cols)}): {obj_cols}')

# Encode: OOF target encoding + frequency + ordinal
SMOOTH_4 = 5
global_mean = float(y_trn.mean())
te_maps = {}
freq_maps = {}

for col in obj_cols:
    for frame in [X_trn, X_val, X_sub]:
        frame[col] = frame[col].fillna('__NA__').astype(str)

    # OOF target encoding on training split
    X_trn[f'{col}_te'] = global_mean
    skf4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for ti, vi in skf4.split(X_trn, y_trn):
        cats = X_trn[col].iloc[ti]; ys = y_trn.iloc[ti]
        agg = pd.DataFrame({'s': ys.groupby(cats).sum(), 'c': ys.groupby(cats).count()})
        sm = (agg['s'] + SMOOTH_4 * global_mean) / (agg['c'] + SMOOTH_4)
        X_trn.iloc[vi, X_trn.columns.get_loc(f'{col}_te')] = (
            X_trn[col].iloc[vi].map(sm).fillna(global_mean).values)

    # Full map for val + test
    full_agg = pd.DataFrame({'s': y_trn.groupby(X_trn[col]).sum(),
                             'c': y_trn.groupby(X_trn[col]).count()})
    full_sm = (full_agg['s'] + SMOOTH_4 * global_mean) / (full_agg['c'] + SMOOTH_4)
    te_maps[col] = full_sm.to_dict()
    X_val[f'{col}_te'] = X_val[col].map(te_maps[col]).fillna(global_mean)
    X_sub[f'{col}_te'] = X_sub[col].map(te_maps[col]).fillna(global_mean)

    # Frequency encoding
    fmap = X_trn[col].value_counts().to_dict()
    freq_maps[col] = fmap
    X_trn[f'{col}_freq'] = X_trn[col].map(fmap).fillna(0).astype(int)
    X_val[f'{col}_freq'] = X_val[col].map(fmap).fillna(0).astype(int)
    X_sub[f'{col}_freq'] = X_sub[col].map(fmap).fillna(0).astype(int)

# Ordinal encoding
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
oe_trn = pd.DataFrame(oe.fit_transform(X_trn[obj_cols]),
                       columns=[f'{c}_oe' for c in obj_cols], index=X_trn.index)
oe_val = pd.DataFrame(oe.transform(X_val[obj_cols]),
                       columns=[f'{c}_oe' for c in obj_cols], index=X_val.index)
oe_sub = pd.DataFrame(oe.transform(X_sub[obj_cols]),
                       columns=[f'{c}_oe' for c in obj_cols], index=X_sub.index)
X_trn = pd.concat([X_trn.drop(columns=obj_cols), oe_trn], axis=1)
X_val = pd.concat([X_val.drop(columns=obj_cols), oe_val], axis=1)
X_sub = pd.concat([X_sub.drop(columns=obj_cols), oe_sub], axis=1)

for c in X_trn.columns:
    X_trn[c] = pd.to_numeric(X_trn[c], errors='coerce')
    X_val[c] = pd.to_numeric(X_val[c], errors='coerce')
    X_sub[c] = pd.to_numeric(X_sub[c], errors='coerce')

# Align test columns
for c in X_trn.columns:
    if c not in X_sub.columns:
        X_sub[c] = 0
X_sub = X_sub[X_trn.columns]

print(f'Feature matrix — train: {X_trn.shape}, val: {X_val.shape}, test: {X_sub.shape}')
print(f'Target balance (train): {y_trn.value_counts(normalize=True).to_string()}')

# Scaled versions for LR / MLP (NaN filled for linear models)
scaler = StandardScaler()
X_trn_sc = pd.DataFrame(scaler.fit_transform(X_trn.fillna(0)),
                         columns=X_trn.columns, index=X_trn.index)
X_val_sc = pd.DataFrame(scaler.transform(X_val.fillna(0)),
                         columns=X_val.columns, index=X_val.index)
X_sub_sc = pd.DataFrame(scaler.transform(X_sub.fillna(0)),
                         columns=X_sub.columns, index=X_sub.index)
print('StandardScaler applied (separate scaled copies for LR/MLP)')

In [ ]:
# --- Evaluation helpers ---

def find_best_threshold(y_true, probs, lo=0.05, hi=0.75, step=0.005):
    best_ba, best_t = 0, 0.5
    for t in np.arange(lo, hi, step):
        ba = balanced_accuracy_score(y_true, (probs >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    return best_t, best_ba

def eval_model(name, y_true, probs):
    t, ba = find_best_threshold(y_true, probs)
    preds = (probs >= t).astype(int)
    auc = roc_auc_score(y_true, probs)
    print(f'\n[{name}] Best threshold={t:.3f}')
    print(f'Balanced Accuracy : {ba:.4f}')
    print(f'ROC-AUC           : {auc:.4f}')
    print(classification_report(y_true, preds))
    return t, ba, auc, probs

### Model 1: Baseline — Logistic Regression
A simple linear baseline to establish the performance floor. `class_weight='balanced'` internally adjusts to our ~6% minority class.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                        class_weight='balanced', n_jobs=-1)
lr.fit(X_trn_sc, y_trn)

lr_probs = lr.predict_proba(X_val_sc)[:, 1]
lr_t, lr_ba, lr_auc, _ = eval_model('Logistic Regression', y_val, lr_probs)

### Model 2: XGBoost — GPU-Accelerated Gradient Boosting
XGBoost uses second-order Taylor expansion of the loss function and native NaN handling.
The `device='cuda'` parameter offloads tree construction to the GPU.
Early stopping on a held-out validation set prevents overfitting.

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=8000,
    max_depth=8,
    learning_rate=0.005,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=10,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    early_stopping_rounds=150,
)

# Train on FULL X_trn, monitor early stopping on X_val
xgb_model.fit(X_trn, y_trn,
              eval_set=[(X_val, y_val)],
              verbose=200)
print(f'XGBoost stopped at iteration {xgb_model.best_iteration}')

xgb_probs = xgb_model.predict_proba(X_val)[:, 1]
xgb_t, xgb_ba, xgb_auc, _ = eval_model('XGBoost', y_val, xgb_probs)

### Model 3: PyTorch Feed-Forward MLP (GPU)
4-layer MLP (input → 256 → 128 → 64 → 1) with:
- **Dropout (p=0.3)** between hidden layers
- **Batch Normalization** for stable activations
- **L2 weight decay** via AdamW
- **Class-weighted BCE loss** — one minority sample worth ~15× a majority sample
- **Cosine annealing** learning rate schedule + early stopping

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class WildlifeMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

n_features = X_trn_sc.shape[1]
mlp = WildlifeMLP(n_features).to(device)

neg_count = (y_trn == 0).sum()
pos_count = (y_trn == 1).sum()
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=5e-4, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

X_tr_t = torch.tensor(X_trn_sc.values, dtype=torch.float32)
y_tr_t = torch.tensor(y_trn.values, dtype=torch.float32).unsqueeze(1)
X_va_t = torch.tensor(X_val_sc.values, dtype=torch.float32)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)

best_auc = 0
patience, wait = 15, 0

for epoch in range(120):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    mlp.eval()
    with torch.no_grad():
        val_logits = mlp(X_va_t.to(device)).cpu()
        val_probs = torch.sigmoid(val_logits).numpy().ravel()
        auc = roc_auc_score(y_val, val_probs)
    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | AUC={auc:.4f} | best={best_auc:.4f} | lr={scheduler.get_last_lr()[0]:.6f}')
    if wait >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

mlp.load_state_dict(best_state)
mlp.eval()
with torch.no_grad():
    mlp_probs = torch.sigmoid(mlp(X_va_t.to(device)).cpu()).numpy().ravel()

mlp_t, mlp_ba, mlp_auc, _ = eval_model('PyTorch MLP', y_val, mlp_probs)

### Model 4: HistGradientBoosting (sklearn)
Complementary tree ensemble using histogram-based splits. Supports native NaN handling
and built-in early stopping.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate=0.02, max_iter=5000, max_depth=8,
    max_leaf_nodes=127, min_samples_leaf=20,
    l2_regularization=0.1,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=50, random_state=42
)
hgb.fit(X_trn, y_trn)
print(f'HGB stopped at {hgb.n_iter_} iterations')

hgb_probs = hgb.predict_proba(X_val)[:, 1]
hgb_t, hgb_ba, hgb_auc, _ = eval_model('HistGradientBoosting', y_val, hgb_probs)

### Model 5: LightGBM (GPU)
LightGBM uses leaf-wise tree growth (vs XGBoost's level-wise), which often finds better splits faster. It handles categorical features natively and typically outperforms XGBoost on tabular data. Adding it provides critical ensemble diversity.

In [ ]:
import lightgbm as lgb

neg_c = (y_trn == 0).sum()
pos_c = (y_trn == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=8000,
    max_depth=-1,
    num_leaves=127,
    learning_rate=0.005,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_samples=20,
    scale_pos_weight=neg_c / pos_c,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    device='gpu',
    verbose=-1,
)

lgb_model.fit(
    X_trn, y_trn,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(150, verbose=True),
        lgb.log_evaluation(200),
    ],
)
print(f'LightGBM stopped at iteration {lgb_model.best_iteration_}')

lgb_probs = lgb_model.predict_proba(X_val)[:, 1]
lgb_t, lgb_ba, lgb_auc, _ = eval_model('LightGBM', y_val, lgb_probs)

### XGBoost Hyperparameter Tuning (GPU)
RandomizedSearchCV over a wide parameter space targeting balanced accuracy.
Uses a 60k subsample for speed with 3-fold stratified CV.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
 

param_grid = {
    'n_estimators': [800, 1200, 2000],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.005, 0.01, 0.02, 0.03],
    'scale_pos_weight': [1, 5, 10, 15],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [5, 10, 20],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1.0, 2.0],
}

rng = np.random.RandomState(42)
n_sub = min(50_000, len(X_trn))
idx = rng.choice(len(X_trn), size=n_sub, replace=False)
X_rs = X_trn.fillna(0).iloc[idx]
y_rs = y_trn.iloc[idx]

print(f'Running RandomizedSearchCV (30 iter x 3-fold on {n_sub:,} samples) ...')
rs = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='auc',
                       tree_method='hist', device='cuda'),
    param_grid,
    n_iter=30, cv=3, scoring='balanced_accuracy',
    random_state=42, n_jobs=1, verbose=1
)
rs.fit(X_rs, y_rs)

print(f'\nBest CV balanced accuracy: {rs.best_score_:.4f}')
print(f'Best params: {rs.best_params_}')

xgb_tuned = xgb.XGBClassifier(**rs.best_params_, random_state=42, n_jobs=-1,
                                eval_metric='auc', tree_method='hist', device='cuda')
xgb_tuned.fit(X_trn, y_trn)

xgbt_probs = xgb_tuned.predict_proba(X_val)[:, 1]
xgbt_t, xgbt_ba, xgbt_auc, _ = eval_model('XGBoost (tuned)', y_val, xgbt_probs)

### K-Fold OOF Stacking Ensemble

Instead of a simple weighted blend, we use **stacking** — a two-level ensemble:

1. **Level 0 (base models):** 5-fold CV trains all 6 models on each fold, collecting out-of-fold (OOF) predictions for all 307K training rows. Tree models (XGBoost, LightGBM) are trained with 3 random seeds and averaged to reduce variance.
2. **Level 1 (meta-learner):** A LogisticRegression is trained on the 6 OOF probability columns to learn optimal blending — including interactions between models.
3. **Threshold:** Optimized on all 307K OOF predictions (much more stable than a single 20% split).
4. **Submission:** All base models are retrained on 100% of training data, then predictions are fed through the meta-learner.

In [ ]:
# Quick single-split comparison (for reference only — stacking cell below is the real ensemble)
print(f'{"MODEL":25s} | {"Val Bal.Acc":>12s} | {"Val AUC":>10s}')
print('-' * 55)
for name, probs in [('LR', lr_probs), ('XGBoost', xgb_probs), ('XGBoost tuned', xgbt_probs),
                     ('MLP', mlp_probs), ('HGB', hgb_probs), ('LightGBM', lgb_probs)]:
    t, ba = find_best_threshold(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    print(f'{name:25s} | {ba:>12.4f} | {auc:>10.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  K-Fold OOF Stacking Ensemble
#  - 5-fold CV generates OOF predictions from all base models
#  - Multi-seed averaging for tree models (3 seeds) reduces variance
#  - LogisticRegression meta-learner on stacked OOF probabilities
#  - Threshold optimized on ALL 307K OOF predictions (stable)
# ═══════════════════════════════════════════════════════════════════
import time

N = len(y_all)
n_folds = 5
SEEDS = [42, 123, 7]

oof_preds = np.zeros((N, 6), dtype=np.float64)
stack_model_names = ['LR', 'XGB', 'HGB', 'LGB', 'MLP', 'XGB_tuned']

skf_stack = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

t0 = time.time()
for fold_i, (trn_idx, val_idx) in enumerate(skf_stack.split(X_all, y_all)):
    print(f'\n{"="*50} FOLD {fold_i+1}/{n_folds} {"="*50}')
    Xf_trn = X_all.iloc[trn_idx]
    Xf_val = X_all.iloc[val_idx]
    yf_trn = y_all.iloc[trn_idx]
    yf_val = y_all.iloc[val_idx]

    sc_f = StandardScaler()
    Xf_trn_sc = pd.DataFrame(sc_f.fit_transform(Xf_trn.fillna(0)),
                              columns=Xf_trn.columns, index=Xf_trn.index)
    Xf_val_sc = pd.DataFrame(sc_f.transform(Xf_val.fillna(0)),
                              columns=Xf_val.columns, index=Xf_val.index)

    # 0: LR
    lr_f = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                              class_weight='balanced', n_jobs=-1)
    lr_f.fit(Xf_trn_sc, yf_trn)
    oof_preds[val_idx, 0] = lr_f.predict_proba(Xf_val_sc)[:, 1]
    print(f'  LR done')

    # 1: XGBoost (multi-seed average)
    xgb_acc = np.zeros(len(val_idx))
    for s in SEEDS:
        m = xgb.XGBClassifier(n_estimators=5000, max_depth=8, learning_rate=0.01,
                               subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1,
                               reg_lambda=1.0, min_child_weight=10, random_state=s,
                               n_jobs=-1, eval_metric='auc', tree_method='hist',
                               device='cuda', early_stopping_rounds=100)
        m.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)], verbose=0)
        xgb_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 1] = xgb_acc / len(SEEDS)
    print(f'  XGB (3 seeds) done')

    # 2: HGB
    hgb_f = HistGradientBoostingClassifier(
        learning_rate=0.02, max_iter=5000, max_depth=8, max_leaf_nodes=127,
        min_samples_leaf=20, l2_regularization=0.1, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=50, random_state=42)
    hgb_f.fit(Xf_trn, yf_trn)
    oof_preds[val_idx, 2] = hgb_f.predict_proba(Xf_val)[:, 1]
    print(f'  HGB done')

    # 3: LightGBM (multi-seed average)
    neg_f = (yf_trn == 0).sum(); pos_f = (yf_trn == 1).sum()
    lgb_acc = np.zeros(len(val_idx))
    for s in SEEDS:
        m = lgb.LGBMClassifier(n_estimators=5000, num_leaves=127, learning_rate=0.01,
                                subsample=0.8, colsample_bytree=0.7, min_child_samples=20,
                                scale_pos_weight=neg_f/pos_f, reg_alpha=0.1, reg_lambda=1.0,
                                random_state=s, n_jobs=-1, device='gpu', verbose=-1)
        m.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 3] = lgb_acc / len(SEEDS)
    print(f'  LGB (3 seeds) done')

    # 4: MLP
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    nf = Xf_trn_sc.shape[1]
    mlp_f = WildlifeMLP(nf).to(dev)
    pw = torch.tensor([(yf_trn==0).sum() / (yf_trn==1).sum()], dtype=torch.float32).to(dev)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt_m = torch.optim.AdamW(mlp_f.parameters(), lr=5e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_m, T_0=20, T_mult=2)
    Xt = torch.tensor(Xf_trn_sc.values, dtype=torch.float32)
    yt = torch.tensor(yf_trn.values, dtype=torch.float32).unsqueeze(1)
    Xvt = torch.tensor(Xf_val_sc.values, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt, yt), batch_size=1024, shuffle=True)
    best_auc_f, wait_f, best_st = 0, 0, None
    for ep in range(80):
        mlp_f.train()
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            opt_m.zero_grad(); crit(mlp_f(xb), yb).backward(); opt_m.step()
        sched.step()
        mlp_f.eval()
        with torch.no_grad():
            vp = torch.sigmoid(mlp_f(Xvt.to(dev)).cpu()).numpy().ravel()
            af = roc_auc_score(yf_val, vp)
        if af > best_auc_f:
            best_auc_f = af; best_st = {k: v.cpu().clone() for k, v in mlp_f.state_dict().items()}; wait_f = 0
        else:
            wait_f += 1
        if wait_f >= 10: break
    mlp_f.load_state_dict(best_st); mlp_f.eval()
    with torch.no_grad():
        oof_preds[val_idx, 4] = torch.sigmoid(mlp_f(Xvt.to(dev)).cpu()).numpy().ravel()
    print(f'  MLP done (ep={ep+1})')

    # 5: XGB tuned (multi-seed, uses best params from RandomizedSearchCV)
    xgbt_acc = np.zeros(len(val_idx))
    bp = dict(rs.best_params_)
    bp.pop('n_estimators', None)
    for s in SEEDS:
        m = xgb.XGBClassifier(**bp, n_estimators=5000, random_state=s, n_jobs=-1,
                               eval_metric='auc', tree_method='hist', device='cuda',
                               early_stopping_rounds=100)
        m.fit(Xf_trn, yf_trn, eval_set=[(Xf_val, yf_val)], verbose=0)
        xgbt_acc += m.predict_proba(Xf_val)[:, 1]
    oof_preds[val_idx, 5] = xgbt_acc / len(SEEDS)
    print(f'  XGB_tuned (3 seeds) done')

    print(f'  Fold {fold_i+1} elapsed: {(time.time()-t0)/60:.1f} min')

# --- OOF per-model evaluation ---
print(f'\n{"="*60}')
print(f'{"MODEL":25s} | {"OOF Bal.Acc":>12s} | {"OOF AUC":>10s}')
print(f'{"-"*60}')
for i, name in enumerate(stack_model_names):
    t_i, ba_i = find_best_threshold(y_all, oof_preds[:, i])
    auc_i = roc_auc_score(y_all, oof_preds[:, i])
    print(f'{name:25s} | {ba_i:>12.4f} | {auc_i:>10.4f}')

# --- Stacking meta-learner ---
meta_lr = LogisticRegression(solver='saga', max_iter=500, class_weight='balanced',
                             C=1.0, random_state=42)
meta_lr.fit(oof_preds, y_all)
oof_meta_probs = meta_lr.predict_proba(oof_preds)[:, 1]

best_ens_t, best_ens_ba = find_best_threshold(y_all, oof_meta_probs, step=0.001)
oof_meta_auc = roc_auc_score(y_all, oof_meta_probs)

print(f'\n{"STACKED ENSEMBLE":25s} | {best_ens_ba:>12.4f} | {oof_meta_auc:>10.4f}  <<<')
print(f'{"="*60}')
print(f'Optimal threshold: {best_ens_t:.3f}')
print(classification_report(y_all, (oof_meta_probs >= best_ens_t).astype(int)))
print(f'Total stacking time: {(time.time()-t0)/60:.1f} min')

## Phase 5: Generate Competition Submission

Retrain every base model on **100% of training data** (no hold-out), generate test
predictions, feed through the stacking meta-learner, and apply the OOF-optimized threshold.
This gives ~2% higher accuracy than using models trained on only 80%.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Retrain ALL base models on 100% training data, then predict test
#  via the fitted stacking meta-learner + optimized threshold.
# ═══════════════════════════════════════════════════════════════════
print('Retraining all models on 100% training data for final submission...\n')
t0_sub = time.time()

# Scaled full training set + test set
sc_full = StandardScaler()
X_all_sc = pd.DataFrame(sc_full.fit_transform(X_all.fillna(0)),
                         columns=X_all.columns, index=X_all.index)
X_sub_sc_full = pd.DataFrame(sc_full.transform(X_sub.fillna(0)),
                              columns=X_sub.columns, index=X_sub.index)

test_base_preds = np.zeros((len(X_sub), 6), dtype=np.float64)
neg_all = (y_all == 0).sum(); pos_all = (y_all == 1).sum()

# 0: LR
lr_full = LogisticRegression(solver='saga', max_iter=300, random_state=42,
                              class_weight='balanced', n_jobs=-1)
lr_full.fit(X_all_sc, y_all)
test_base_preds[:, 0] = lr_full.predict_proba(X_sub_sc_full)[:, 1]
print('  LR retrained')

# 1: XGB (multi-seed)
for s in SEEDS:
    m = xgb.XGBClassifier(n_estimators=3000, max_depth=8, learning_rate=0.01,
                           subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1,
                           reg_lambda=1.0, min_child_weight=10, random_state=s,
                           n_jobs=-1, eval_metric='auc', tree_method='hist', device='cuda')
    m.fit(X_all, y_all, verbose=0)
    test_base_preds[:, 1] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 1] /= len(SEEDS)
print('  XGB retrained (3 seeds)')

# 2: HGB
hgb_full = HistGradientBoostingClassifier(
    learning_rate=0.02, max_iter=3000, max_depth=8, max_leaf_nodes=127,
    min_samples_leaf=20, l2_regularization=0.1, random_state=42)
hgb_full.fit(X_all, y_all)
test_base_preds[:, 2] = hgb_full.predict_proba(X_sub)[:, 1]
print('  HGB retrained')

# 3: LGB (multi-seed)
for s in SEEDS:
    m = lgb.LGBMClassifier(n_estimators=3000, num_leaves=127, learning_rate=0.01,
                            subsample=0.8, colsample_bytree=0.7, min_child_samples=20,
                            scale_pos_weight=neg_all/pos_all, reg_alpha=0.1, reg_lambda=1.0,
                            random_state=s, n_jobs=-1, device='gpu', verbose=-1)
    m.fit(X_all, y_all)
    test_base_preds[:, 3] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 3] /= len(SEEDS)
print('  LGB retrained (3 seeds)')

# 4: MLP
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mlp_full = WildlifeMLP(X_all_sc.shape[1]).to(dev)
pw = torch.tensor([neg_all / pos_all], dtype=torch.float32).to(dev)
crit = nn.BCEWithLogitsLoss(pos_weight=pw)
opt_m = torch.optim.AdamW(mlp_full.parameters(), lr=5e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_m, T_0=20, T_mult=2)
Xt = torch.tensor(X_all_sc.values, dtype=torch.float32)
yt = torch.tensor(y_all.values, dtype=torch.float32).unsqueeze(1)
dl = DataLoader(TensorDataset(Xt, yt), batch_size=1024, shuffle=True)
for ep in range(60):
    mlp_full.train()
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt_m.zero_grad(); crit(mlp_full(xb), yb).backward(); opt_m.step()
    sched.step()
mlp_full.eval()
Xst = torch.tensor(X_sub_sc_full.values, dtype=torch.float32)
with torch.no_grad():
    test_base_preds[:, 4] = torch.sigmoid(mlp_full(Xst.to(dev)).cpu()).numpy().ravel()
print(f'  MLP retrained (60 epochs)')

# 5: XGB tuned (multi-seed)
bp = dict(rs.best_params_)
bp.pop('n_estimators', None)
for s in SEEDS:
    m = xgb.XGBClassifier(**bp, n_estimators=3000, random_state=s, n_jobs=-1,
                           eval_metric='auc', tree_method='hist', device='cuda')
    m.fit(X_all, y_all, verbose=0)
    test_base_preds[:, 5] += m.predict_proba(X_sub)[:, 1]
test_base_preds[:, 5] /= len(SEEDS)
print('  XGB_tuned retrained (3 seeds)')

# --- Apply stacking meta-learner + threshold ---
sub_meta_probs = meta_lr.predict_proba(test_base_preds)[:, 1]
sub_preds = (sub_meta_probs >= best_ens_t).astype(int)

submission = pd.DataFrame({
    'INDEX_NR': submission_ids,
    'INDICATED_DAMAGE': sub_preds
})
submission.to_csv('submission.csv', index=False)

print(f'\nsubmission.csv saved  ({len(submission):,} rows)')
print(f'Predicted damage rate: {sub_preds.mean():.4f}')
print(f'Distribution: 0={int((sub_preds==0).sum()):,}  1={int((sub_preds==1).sum()):,}')
print(f'Retrain time: {(time.time()-t0_sub)/60:.1f} min')
print(f'\nFirst 10 rows:\n{submission.head(10).to_string(index=False)}')

try:
    colab_files.download('submission.csv')
except Exception:
    print('\n(Auto-download skipped — file saved to working directory.)')